# 01 · API와 메시지
**2세션 · API와 메시지** — 실습 25분

1. 첫 호출 — SDK를 그대로 써 본다
2. 응답 읽기 — `stop_reason` · `usage`
3. system과 user
4. 모델은 기억하지 않는다 — 대화 이력
5. 같은 질문 다섯 번 — 그리고 temperature
6. 토큰과 비용

**읽는 법** — 실습 셀마다 바로 앞에 **▶ 이 셀에서 할 것**, 바로 뒤에 **✔ 이 셀의 시사점**이 있습니다. 셀을 돌리기 전에 ▶를, 돌린 뒤에 출력과 ✔를 같이 읽습니다.

**▶ 이 셀에서 할 것** — 준비. 공용 도구 `llm`(`ETRI_LLM_3days/common/llm.py`)을 불러오고, 어떤 모델로 부르는지 · 실제 호출인지 모의 모드인지 확인합니다.

In [ ]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")          # day 폴더로 이동 — data/ · prompts/ · src/ 가 여기 있다

from common import llm      # 공용 API 도구 (ETRI_LLM_3days/common/llm.py)
if llm.MOCK:
    print("모델:", llm.MODEL, "| 모의 모드")
else:
    print("모델:", llm.MODEL, "| 실제 호출")

**✔ 이 셀의 시사점**
- 출력의 모델 이름이 오늘 쓸 모델인지 확인합니다. 다르면 `.env`의 `MODEL`을 고칩니다.
- `실제 호출`이어야 이 노트북의 숫자(토큰 수 · 비용)가 의미가 있습니다. `모의 모드`면 답이 가짜 문장으로 나옵니다.

## 1. 첫 호출
`llm.call()`은 아래 SDK 코드를 감싼 것입니다. 감싸기 전 모습을 한 번은 그대로 봐 둡니다.

**▶ 이 셀에서 할 것** — `llm` 도구 없이 anthropic SDK로 직접 한 번 호출합니다. 답 글을 찍고, 이어서 돌아온 `r`을 **통째로** 찍어 글 말고 무엇이 들어 있는지 봅니다. (모의 모드에서는 건너뜁니다.)

In [ ]:
if not llm.MOCK:
    import anthropic
    client = anthropic.Anthropic()          # .env의 ANTHROPIC_API_KEY를 읽는다
    r = client.messages.create(
        model=llm.MODEL,
        max_tokens=1500,
        messages=[{"role": "user", "content": "연차 신청 절차를 세 단계로 알려줘"}],
    )
    for block in r.content:                 # 답은 블록 여러 개로 올 수 있다
        if block.type == "text":
            print(block.text)
        else:
            print(f"[{block.type} 블록]")
    print("\n--- r에 저장된 것 전체 (글은 그중 하나일 뿐) ---")
    print(r)
else:
    print("모의 모드 — 건너뜀")

**✔ 이 셀의 시사점**
- `r`은 글 하나가 아니라 `id`·`role`·`type`·`content`(블록 리스트)·`stop_reason`·`stop_sequence`·`model`·`usage`(입력/출력 토큰 · 캐시 사용량 · 처리 등급 등)·`container`·`diagnostics`·`stop_details`까지 다 들어 있는 객체입니다.
- 답 글은 `content` 리스트 안의 **text 블록**에 들어 있습니다. 위 셀은 그 글만 꺼내 찍었습니다. text 블록 앞에 `[thinking 블록]`이 찍혔다면 그게 무엇인지는 2절에서 봅니다. 나머지 항목은 2·4절에서 하나씩 다룹니다.
- 앞으로는 이 호출을 `llm.call()` 한 줄로 씁니다. 하는 일은 같고, 자주 보는 값(글 · 멈춘 이유 · 토큰 수 · 걸린 시간)을 바로 꺼내 쓰게 해 둔 것입니다.

## 2. 응답 읽기
응답에는 글 말고도 **왜 멈췄는지**(`stop_reason`)와 **토큰을 얼마나 썼는지**(`usage`)가 들어 있습니다.

**▶ 이 셀에서 할 것** — 1절과 같은 질문을 이번엔 `llm.call()`로 보내고, 답 글과 함께 받은 블록의 종류 · `stop_reason` · 입력 토큰 · 출력 토큰 · 걸린 시간을 찍습니다.

In [ ]:
r = llm.call("연차 신청 절차를 세 단계로 알려줘", max_tokens=1500)
print(r.text)
print("---")
print("받은 블록   :", [block.type for block in r.raw.content])
print("stop_reason :", r.stop_reason)
print("입력 토큰   :", r.input_tokens)
print("출력 토큰   :", r.output_tokens)
print("걸린 시간   :", f"{r.seconds:.2f}초")

**✔ 이 셀의 시사점**
- `stop_reason`이 `end_turn`이면 모델이 스스로 답을 끝낸 것입니다. (끝내지 못하고 잘린 경우는 2절 끝에서 봅니다.)
- 비용은 **입력 토큰과 출력 토큰** 두 숫자로 계산됩니다(6절). 그래서 이 두 숫자를 늘 같이 봅니다.
- **받은 블록에 `thinking`이 있으면**, 모델이 답 글을 쓰기 전에 **보이지 않는 생각**을 먼저 한 것입니다. Haiku 5.5는 생각할지 말지를 스스로 정해서, 같은 질문이라도 생각할 때가 있고 안 할 때가 있습니다. 위 출력에 `thinking`이 없었다면 2절 끝 "max_tokens에서 잘리면" 셀에서 보게 될 수 있습니다.
  - 생각한 내용은 우리에게 빈 칸으로 옵니다. 하지만 **생각에 쓴 토큰도 출력 토큰에 들어가서 돈을 냅니다.** 그래서 출력 토큰이 답 글보다 커집니다. 얼마나 커지는지는 아래 "출력 토큰 중 생각에 쓴 몫"에서 재 봅니다.
  - 생각도 `max_tokens` 안에서 합니다. 그래서 이 노트북은 `max_tokens`를 넉넉히(1500) 줍니다.
- 그런데 입력 토큰 수는 질문 글자 수와 맞지 않습니다. 왜 그런지 아래에서 "안녕" 두 글자로 따져 봅니다.

### 입력은 두 번 모양이 바뀐다 — 내 코드 → 서버 → 모델
"안녕"이 모델에 닿기까지 **두 단계**를 거치고, 단계마다 모양이 다릅니다. 이 둘을 섞으면 토큰 수가 이해가 안 됩니다.

```
내 코드 ──(JSON)──▶ Anthropic 서버 ──(토큰 열)──▶ 모델
```

**1단계 — 내 코드가 서버로 보내는 것 (JSON).** 내가 입력한 건 "안녕" 두 글자지만, 실제로 나가는 건 JSON 덩어리입니다.
이 JSON을 받는 쪽은 **모델이 아니라 Anthropic 서버**입니다. 모델은 `{` · `"role"` · `"user"` 같은 JSON 글자를 하나도 보지 않습니다 — 서버가 JSON을 풀어서 모델용 입력으로 다시 만들어 넘기기 때문입니다(2단계).

**▶ 이 셀에서 할 것** — 실제로 보내지는 않고, "안녕"을 보낼 때 서버로 나가는 요청(JSON)의 모양만 찍어 봅니다.

In [ ]:
import json

# 보내지는 않고, 보낼 요청의 모양만 찍어 본다
request = {
    "model": llm.MODEL,
    "max_tokens": 20,
    "messages": [{"role": "user", "content": "안녕"}],
}
print("내가 입력한 건: '안녕' (2글자)")
print("\n실제로 서버에 보내는 요청 (JSON):")
print(json.dumps(request, ensure_ascii=False, indent=2))

**✔ 이 셀의 시사점**
- 내가 쓴 건 2글자인데, 나가는 건 `model` · `max_tokens` · `messages` 세 항목짜리 JSON입니다.
- 그럼 이 JSON 글자가 전부 토큰으로 세어질까요? 아닙니다 — 아래 2단계에서 서버가 이걸 어떻게 바꾸는지 봅니다.

**2단계 — 서버가 JSON을 풀어서 모델 입력으로 바꾼다.** 세 항목이 각각 이렇게 쓰입니다.
- `model` → 어느 모델에 보낼지 **고르는 데만** 쓰고 버림. 모델 입력에 안 들어감
- `max_tokens` → 생성을 몇 토큰에서 **끊을지**에만 쓰고 버림. 모델 입력에 안 들어감
- `messages` → **이것만** 모델 입력이 됨. 단, `"role": "user"` 같은 글자 그대로가 아니라 "여기서부터 user의 말" 같은 뜻의 **특수 토큰**(구조 표식)으로 바뀜

그래서 모델이 읽는 것은 JSON이 아니라 대략 이런 토큰 열입니다 (`[ ]`는 글자가 아닌 특수 토큰):
```
[대화 시작] [user 차례] 안녕 [내용 끝] [assistant 차례] ← 모델은 여기서부터 이어 쓴다
```

> **어디까지가 확인된 것인가.** 이 토큰 열의 정확한 모양은 공개된 문서에 없어서 우리가 직접 볼 수 없습니다. 아래에서 직접 재서 확인하는 것은 두 가지입니다 — ① JSON 글자는 토큰으로 세어지지 않는다 ② 내용 말고 붙는 표식의 **개수**와, 메시지가 늘 때 몇 개씩 느는지. 표식 하나하나의 이름(`[대화 시작]` 등)은 그 개수에 맞춰 그린 **그림**입니다.

**▶ 이 셀에서 할 것** — "안녕"을 실제로 보내서 입력 토큰이 몇 개로 세어지는지 보고, 돌아온 응답 원본(`r.raw`)을 통째로 찍습니다.

In [ ]:
r = llm.call("안녕", max_tokens=20)
print("모델이 받은 입력 토큰 수:", r.input_tokens, " — 내가 쓴 건 '안녕' 2글자")
print("\n돌아온 응답 전체:")
print(r.raw)

**✔ 이 셀의 시사점**
- 2글자를 보냈는데 입력 토큰은 11개입니다. 내가 쓴 글자 말고도 뭔가가 붙어서 세어진다는 뜻입니다.
- 응답에는 글 말고도 `id`(요청 번호) · `stop_reason`(멈춘 이유) · `usage`(토큰 수뿐 아니라 캐시 사용량 · 처리 등급 · 추론 지역까지)가 같이 옵니다. `llm.call()`의 결과는 그중 당장 필요한 몇 개(`text`·`stop_reason`·토큰 수)만 추려 둔 것이고, 전부 보려면 `r.raw`를 엽니다.

### 그 11토큰은 뭔가 — JSON 글자가 아니다
"JSON에서 안녕을 뺀 나머지가 토큰 아니냐"고 생각하기 쉽습니다. 아닙니다. 내용을 바꿔 가며 세어 보면 알 수 있습니다.

**▶ 이 셀에서 할 것** — 모델을 부르지 않고 토큰 수만 세는 `llm.count_tokens()`로, 메시지 내용을 `a` · `hello` · `안` · `안녕`으로 바꿔 가며 셉니다. 마지막엔 JSON 글자 `{"role": "user", "content": ""}` 자체를 내용으로 넣어 셉니다.

In [ ]:
for t in ["a", "hello", "안", "안녕"]:
    print(f"{t!r:34} → {llm.count_tokens(t):>2} 토큰")
j = '{"role": "user", "content": ""}'
print(f"{j!r:34} → {llm.count_tokens(j):>2} 토큰   ← JSON 글자를 내용으로 보내 보면")

**✔ 이 셀의 시사점**
- 한 글자짜리 `a`도 **9토큰**입니다. 내용은 적어도 1토큰이니, 내용과 상관없이 **8개 안팎이 늘 붙는다**는 뜻입니다.
- `hello`는 10, `안`은 9, `안녕`은 11입니다. 글자 수나 단어 수로는 토큰 수를 맞힐 수 없습니다 — 직접 세어 봐야 압니다.
- JSON 글자 `{"role": "user", "content": ""}`를 **내용으로** 넣으면 21토큰, `a`보다 12개나 많습니다. JSON 글자는 그 자체로 토큰을 이만큼 먹습니다. 그런데 `a` 하나를 보낼 때 붙는 건 8개 안팎뿐이니, **늘 붙는 그 8개는 JSON 글자가 아닙니다.**

그러니 "안녕"의 11개는 **"안녕" 내용 + 구조 표식**입니다. `count_tokens`는 합계만 알려 주므로 둘을 몇 대 몇으로 딱 가를 수는 없고, 표식이 정확히 무엇인지도 공개돼 있지 않습니다. 확인되는 것은 **내용과 상관없이 늘 일정한 개수가 붙는다**는 것까지입니다.

### 늘 붙는 개수는 메시지 수에 따라 어떻게 늘어나나
메시지 하나를 감싸는 표식이라면 메시지가 늘 때 같이 늘어야 합니다. 메시지 수를 바꿔 가며 재 봅니다.

**▶ 이 셀에서 할 것** — 메시지를 1개 · 3개 · 5개로 늘려 가며(내용은 모두 한 글자짜리 `a`, `b`) 전체 토큰을 셉니다. 내용을 메시지마다 1토큰으로 보고 뺀 나머지(틀)가 몇 개인지 보고, 마지막에 **틀이 메시지 수에 따라 어떻게 느는지** 식으로 정리합니다.

In [ ]:
import anthropic
client = anthropic.Anthropic()

def count_messages(messages):
    """messages 전체의 입력 토큰 수를 센다 (모델은 부르지 않는다)."""
    result = client.messages.count_tokens(model=llm.MODEL, messages=messages)
    return result.input_tokens

user_a = {"role": "user", "content": "a"}
assistant_b = {"role": "assistant", "content": "b"}

cases = [
    ("메시지 1개 [a]", [user_a]),
    ("메시지 3개 [a|b|a]", [user_a, assistant_b, user_a]),
    ("메시지 5개 [a|b|a|b|a]", [user_a, assistant_b, user_a, assistant_b, user_a]),
]
frames = []                          # 경우마다 틀(전체 − 내용)을 모은다
for name, messages in cases:
    total = count_messages(messages)
    content = len(messages)          # 내용 a · b는 메시지마다 1토큰으로 본다
    frame = total - content
    frames.append(frame)
    print(f"{name:22} → {total:>2} 토큰 = 내용 {content} + 틀 {frame}")

PER_MESSAGE = (frames[1] - frames[0]) // 2    # 메시지가 2개 늘 때 늘어난 틀 ÷ 2 = 메시지 하나마다 붙는 틀
BASE = frames[0] - PER_MESSAGE                 # 메시지 1개일 때 틀에서 그 메시지 몫을 뺀 나머지 = 대화 전체에 한 번 붙는 틀
print(f"\n→ 틀 = {BASE} + {PER_MESSAGE} × 메시지 수")

**✔ 이 셀의 시사점**
- 메시지가 하나 늘 때마다 틀이 **똑같은 개수만큼** 늡니다. 그래서 틀은 `대화 전체에 한 번 붙는 몫 + 메시지마다 붙는 몫 × 메시지 수`로 나뉩니다. 이 모델(Haiku 5.5)에서는 `6 + 2 × 메시지 수`입니다. 이 두 숫자(`BASE` · `PER_MESSAGE`)는 4절에서 다시 씁니다.
- 이 숫자는 **모델마다 다릅니다.** 같은 실험을 이전 모델 Haiku 4.5에서 하면 `4 + 3 × 메시지 수`가 나왔습니다. 공개된 값이 아니어서 이렇게 직접 재서 알아냅니다.
- 이 실험은 `a`·`b`가 각각 1토큰이라고 보고 계산했습니다. 그 가정이 맞는지는 4절에서 실제 대화의 토큰 수와 맞춰 보며 확인합니다.
- 실무에서 기억할 것: 내가 쓴 글자(2개)보다 붙는 표식이 더 크고, 글자 수로는 토큰 수를 어림잡을 수 없습니다. 그리고 4절에서 보겠지만, 턴이 쌓이면 **메시지마다 붙는 몫**도 매번 전부 다시 들어갑니다.

### 출력 토큰 중 생각에 쓴 몫
2절 앞에서 본 것처럼 Haiku 5.5는 답 전에 보이지 않는 생각을 할 수 있고, 그 토큰도 출력 토큰에 들어갑니다. 생각은 `llm.call(..., thinking=...)`으로 끌 수 있습니다. 끄는 값은 모델마다 다르고, 이 모델(Haiku 5.5)은 `{"type": "disabled"}`입니다.

**▶ 이 셀에서 할 것** — 같은 질문을 thinking **기본(켜짐)**과 **끄기**로 두 번씩 보냅니다. 답 글만 따로 센 토큰(바로 위에서 잰 틀을 빼서 구함)을 출력 토큰에서 빼서, 나머지 — 생각에 쓴 몫 — 를 봅니다.

In [ ]:
question = "연차 신청 절차를 세 단계로 자세히 알려줘"
FRAME_ONE = BASE + PER_MESSAGE      # 메시지 1개짜리 틀 (바로 위에서 잰 값) — 글만 셀 때 뺀다

cases = [
    ("기본 (켜짐)", None),
    ("끄기", {"type": "disabled"}),  # 끄는 값은 모델마다 다르다 — 이 값은 Haiku 5.5용
]
for label, thinking in cases:
    for i in range(2):
        r = llm.call(question, max_tokens=1500, thinking=thinking)
        kinds = [block.type for block in r.raw.content]
        text_tokens = llm.count_tokens(r.text) - FRAME_ONE      # 답 글만 따로 센 토큰
        rest = r.output_tokens - text_tokens                    # 출력 토큰 중 글이 아닌 나머지
        print(f"{label:9} {i+1}회 | 블록 {kinds} | 출력 {r.output_tokens:>5} = 글 {text_tokens:>5} + 나머지 {rest:>4}")

**✔ 이 셀의 시사점**
- **끄면** 받은 블록이 `text`뿐이고, 나머지는 2개 안팎입니다. 생각을 안 했는데도 2개쯤 남는 이유는 이 자료로는 알 수 없어서, 그만큼은 오차로 봅니다.
- **켜면** 나머지가 그보다 훨씬 큽니다 — 그것이 **생각에 쓴 토큰**이고, 출력 단가로 돈을 냅니다. 강사가 돌렸을 때 이 질문에서는 약 70~80토큰이었고, 6절의 긴 문서 요약에서는 출력 500토큰 안팎 중 220~360토큰이 생각이었습니다. **얼마나 생각하는지는 질문마다 다릅니다.**
- 강사가 여러 번 돌렸을 때는 켰을 때 **답 글 자체도 300~500토큰쯤 더 길었습니다**(켜면 글 약 1,050~1,320토큰, 끄면 약 740~850토큰). 그래서 켜고 끈 출력 토큰 차이가 전부 생각은 아닙니다 — 글을 따로 세어 봐야 생각의 몫이 보입니다.
- 이 과정은 기본(켜짐) 그대로 씁니다. 끄면 출력 토큰은 줄지만, 생각이 필요한 어려운 질문에서 답이 어떻게 달라지는지는 이 셀로는 알 수 없습니다.

### max_tokens에서 잘리면
`max_tokens`를 작게 주면 답이 중간에 끊기고 `stop_reason`이 `max_tokens`가 됩니다. 생각하는 모델은 생각에 토큰을 먼저 쓰기 때문에, **글은 시작도 못 하고** 끊길 수도 있습니다.

**▶ 이 셀에서 할 것** — "자세히 알려 달라"는 질문을 `max_tokens=20`으로 짧게 묶어 보내고, 받은 블록 · 글 · `stop_reason` · 출력 토큰을 봅니다.

In [ ]:
r = llm.call("연차 신청 절차를 세 단계로 자세히 알려줘", max_tokens=20)
print("받은 블록   :", [block.type for block in r.raw.content])
print("글          :", repr(r.text))
print("stop_reason :", r.stop_reason, "| 출력 토큰:", r.output_tokens)
if r.stop_reason == "max_tokens":
    print("→ 잘렸습니다. max_tokens를 늘리거나 더 짧게 쓰라고 지시합니다.")

**✔ 이 셀의 시사점**
- `stop_reason`이 `max_tokens`이고, 출력 토큰을 20개 다 썼습니다. 글은 쓰다 만 채로 끊기거나, 받은 블록이 `thinking`뿐이고 글이 `''`(빈 문자열)일 수 있습니다 — 20토큰을 **생각하는 데 다 써서** 글을 쓸 차례가 오지 않은 것입니다.
- 어느 쪽이든 **오류는 나지 않습니다** — 그래서 `stop_reason`을 안 보면 잘린 줄 모릅니다.
- 그래서 **프로그램은 답을 쓰기 전에 `stop_reason`부터 확인합니다.** 예를 들어 JSON으로 답하게 했는데 잘리면 파싱이 깨집니다.

## 3. system과 user
매번 같은 **역할 · 규칙 · 형식**은 `system`에, 매번 바뀌는 **자료와 질문**은 `user`에 둡니다.

**▶ 이 셀에서 할 것** — 같은 질문(소방 훈련 안내)을 system 없이 · 인사팀 담당자 · 사내 공지 봇, 세 가지 system으로 보내 답을 나란히 봅니다. 질문은 한 글자도 바꾸지 않습니다.

In [ ]:
question = "다음 주 화요일 오후 2시에 전 직원 소방 훈련이 있습니다. 이 내용을 안내해 주세요."

for system in [
    None,
    "당신은 신입 사원에게 친절하게 설명하는 인사팀 담당자입니다. 두 문장 이내로 씁니다.",
    "당신은 사내 메신저 공지 봇입니다. 이모지 없이 한 줄로, '[공지]'로 시작합니다.",
]:
    r = llm.call(question, system=system, max_tokens=1500)
    print(f"[system] {system}\n{r.text.strip()}\n")

**✔ 이 셀의 시사점**
- 질문은 그대로인데 말투 · 길이 · 형식이 system을 따라 바뀝니다. system에 적은 규칙(두 문장 이내, '[공지]'로 시작 등)이 지켜졌는지 출력에서 직접 확인합니다.
- 그러니 매번 같은 역할 · 규칙 · 형식은 system에 한 번 정해 두고, user에는 그때그때 바뀌는 자료와 질문만 넣습니다.

## 4. 모델은 기억하지 않는다
API는 매 호출이 처음입니다. 이어서 대화하려면 **앞서 주고받은 메시지를 직접 다시 보냅니다** — 그래서 턴이 늘어날수록 **매번 보내는 요청 자체가 커집니다.**

**▶ 이 셀에서 할 것** — 2턴짜리 대화를 돌립니다. 1턴에 이름과 부서를 알려 주고, 2턴에 그걸 다시 묻습니다. 매 턴 **서버로 나가는 요청(JSON) 전체와 돌아오는 응답 전체**를 빠짐없이 찍습니다. (2절에서 봤듯 모델은 이 JSON의 `messages`만 특수 토큰으로 바꿔 받습니다.)

In [ ]:
import json

history = []
sent = []          # 턴마다 (보낸 messages, 응답) 을 남겨 둔다 — 아래에서 토큰 수를 검산한다
turns = [
    "제 이름은 김민수이고, 생산관리팀에서 일합니다.",
    "제 이름과 부서가 뭐였죠?",
]
for i, user_text in enumerate(turns, 1):
    messages = history + [{"role": "user", "content": user_text}]
    request = {"model": llm.MODEL, "max_tokens": 1500, "messages": messages}
    print(f"===== {i}번째 턴 =====")
    print("--- 서버로 나가는 요청 (JSON) ---")
    print(json.dumps(request, ensure_ascii=False, indent=2))

    r = llm.call(user_text, history=history, max_tokens=1500)
    sent.append((messages, r))

    print("\n--- 돌아온 응답 전체 ---")
    print(r.raw)
    print()

    history = messages + [{"role": "assistant", "content": r.text}]

**✔ 이 셀의 시사점**
- **1턴**은 `messages`에 내가 보낸 말 하나뿐이지만, **2턴**은 1턴의 내 말 + 모델의 답 + 이번 내 말, **세 개**가 들어갑니다.
- 2턴에서 모델이 이름과 부서를 답할 수 있는 건 기억해서가 아니라 **1턴 내용을 내가 다시 보냈기 때문**입니다. 이력 없이 "제 이름과 부서가 뭐였죠?"만 보내면 모델은 모릅니다.

### 멀티턴 입력 토큰은 이 식으로 정확히 맞는다
2절 끝에서 잰 틀은 `BASE + PER_MESSAGE × 메시지 수`였습니다(이 모델: `6 + 2 × 메시지 수`). 턴이 쌓이면:

```
input_tokens = BASE + Σ (PER_MESSAGE + 메시지 내용 토큰)     ← 지금까지 쌓인 모든 메시지에 대해
```

**▶ 이 셀에서 할 것** — 방금 돌린 두 턴의 실제 `usage.input_tokens`를 이 식으로 계산한 값과 맞춰 봅니다. 메시지 하나의 내용 토큰은 그 메시지만 따로 센 값에서 메시지 1개짜리 틀(`BASE + PER_MESSAGE`)을 빼서 구합니다.

In [ ]:
FRAME_ONE = BASE + PER_MESSAGE      # 메시지 1개짜리를 셀 때 붙는 틀 (2절에서 잰 값)

for i, (messages, r) in enumerate(sent, 1):
    pieces = []                                                 # 메시지마다 내용 토큰
    for m in messages:
        pieces.append(llm.count_tokens(m["content"]) - FRAME_ONE)   # 그 메시지만 센 값 − 틀 = 내용 토큰
    pred = BASE
    for p in pieces:
        pred = pred + PER_MESSAGE + p
    if pred == r.input_tokens:
        ok = "✓"
    else:
        ok = "✗"
    print(f"{i}턴: 메시지 {len(messages)}개, 내용 토큰 {pieces}")
    print(f"     {BASE} + Σ({PER_MESSAGE}+내용) = {pred}   실제 input_tokens = {r.input_tokens}   {ok}")

**✔ 이 셀의 시사점**
- 두 턴 모두 ✓ — 입력 토큰은 "지금까지 쌓인 메시지 전부의 내용 + 메시지마다 붙는 틀 + 대화 전체에 한 번 붙는 틀"로 정확히 계산됩니다. 2절에서 `a`·`b`를 1토큰으로 본 가정도 맞았다는 뜻입니다.
- 2턴째 내용 토큰 가운데 가장 큰 덩어리는 **1턴에서 모델이 쓴 답 글**입니다. 1턴엔 `output_tokens`로 냈던 것을 2턴엔 `input_tokens`로 **다시** 냅니다 (생각에 쓴 토큰은 이력에 넣지 않으므로 다시 내지 않습니다) — 멀티턴에서 비용이 빨리 불어나는 이유입니다. 메시지마다 붙는 틀도 매번 전부 다시 들어갑니다.

### 요청보다 응답에 훨씬 많은 게 따라온다
요청은 `model` · `max_tokens` · `messages`, 딱 3가지뿐이었습니다.

**▶ 이 셀에서 할 것** — 방금 받은 마지막 응답 원본(`r.raw`)을 항목별로 열어, 각 값과 그 뜻을 한 줄씩 찍어 봅니다.

In [ ]:
u = r.raw.usage
types = [b.type for b in r.raw.content]
block = None
for b in r.raw.content:       # text 블록을 찾는다 — 앞에 thinking 블록이 있을 수 있다
    if b.type == "text":
        block = b
print("요청에는 모델 이름 · max_tokens · messages, 이 3가지만 넣었습니다.")
print("마지막 턴의 응답(r.raw)에는 그보다 훨씬 많은 게 같이 옵니다:\n")
print(f"  content                           : {types}")
print(f"      답이 블록 하나가 아니라 **리스트**인 이유 — 생각(thinking) · 도구 호출(tool_use) ·")
print(f"      글(text)처럼 종류가 다른 블록이 여러 개 올 수 있습니다.")
print(f"  text 블록의 .text                 : {block.text[:40]!r}...")
print(f"      우리가 r.text로 꺼내 쓰는 바로 이 값")
print(f"  text 블록의 .citations            : {block.citations}")
print(f"      문서를 인용하며 답했을 때만 출처가 들어감 (지금은 없음)")
print(f"  role                              : {r.raw.role}")
print(f"      누가 말했는지 — 응답은 항상 'assistant'")
print(f"  type                              : {r.raw.type}")
print(f"      이 객체 자체가 무엇인지 — 'message'")
print(f"  container                         : {r.raw.container}")
print(f"      코드 실행 도구를 쓸 때만 생기는 실행 환경 정보 (안 쓰면 None)")
print(f"  diagnostics                       : {r.raw.diagnostics}")
print(f"      캐시 진단(베타 기능)을 켰을 때만 옴 (안 켜면 None)")
print(f"  stop_details                      : {r.raw.stop_details}")
print(f"      거절(refusal)당했을 때만 이유가 들어감 (지금처럼 정상 답이면 None)")
print(f"  id                                : {r.raw.id}")
print(f"      내가 보낸 적 없음 — 이 응답 하나를 가리키는 고유 번호")
print(f"  stop_reason                       : {r.raw.stop_reason}")
print(f"      내가 물은 적 없음 — 왜 멈췄는지(end_turn · max_tokens · tool_use · ...)")
print(f"  stop_sequence                     : {r.raw.stop_sequence}")
print(f"      어떤 중단 문자열에 걸렸는지 (안 걸리면 None)")
print(f"  model                             : {r.raw.model}")
print(f"      실제로 처리한 모델 — 요청에 쓴 이름과 같아야 정상")
print(f"  usage.input_tokens / output_tokens: {u.input_tokens} / {u.output_tokens}")
print(f"      우리가 Result로 뽑아 쓰는 바로 그 숫자")
print(f"  usage.cache_creation_input_tokens : {u.cache_creation_input_tokens}")
print(f"      프롬프트 캐시에 이번에 새로 쓴 토큰 수 (지금은 캐시를 안 써서 0)")
print(f"  usage.cache_read_input_tokens     : {u.cache_read_input_tokens}")
print(f"      캐시에서 그냥 읽어서 싸게 처리한 토큰 수")
print(f"  usage.service_tier                : {u.service_tier}")
print(f"      이번 요청이 처리된 등급")
print(f"  usage.inference_geo               : {u.inference_geo}")
print(f"      추론이 실제로 일어난 지역")

**✔ 이 셀의 시사점**
- 요청 쪽은 3가지뿐인데, 응답 쪽은 **15가지가 넘는 부가 정보**가 따라옵니다.
- `content`조차 글 하나가 아니라 **블록 리스트**이고, 우리가 늘 쓰는 `r.text`는 그 리스트에서 **text 블록의 글만** 꺼낸 것입니다. thinking 블록은 빠집니다.
- `r.text`만 보면 이 중 거의 전부를 놓칩니다. `llm.call()`의 결과는 당장 쓸모 있는 몇 개만 추려 둔 것이고, 나머지는 `r.raw`를 직접 열어야 보입니다.

## 5. 같은 질문 다섯 번
같은 질문을 두 가지로 각각 5회 보내고 결과를 나란히 놓습니다.

**▶ 이 셀에서 할 것** — 정답이 거의 정해진 질문(연차 신청 절차)과 열린 질문(슬로건)을 각각 5번씩 보냅니다. 매번 출력 토큰 수와 답 앞부분을 찍고, 마지막에 서로 다른 답이 몇 개인지 셉니다.

**관찰할 것**
- 다섯 개 중 몇 개가 서로 같은가
- 달라지는 곳은 문장의 어디인가
- 정답이 거의 정해진 질문(절차)과 열린 질문(슬로건)은 차이가 어떻게 다른가
- 출력 토큰 수는 매번 같은가

In [ ]:
QUESTIONS = {
    "절차": "회사 연차 신청 절차를 세 단계로 알려줘",
    "슬로건": "신제품 무선 청소기 슬로건 하나만 지어줘",
}

runs = {}
for name, q in QUESTIONS.items():
    runs[name] = []
    for i in range(5):
        r = llm.call(q, max_tokens=1500)
        runs[name].append({"text": r.text, "output_tokens": r.output_tokens})
        print(name, i + 1, r.output_tokens, "|", r.text.strip().replace("\n", " ")[:60])
    print()

for name, rs in runs.items():
    print(f"{name}: 서로 다른 답 {len({x['text'] for x in rs})}개 / 5")

**✔ 이 셀의 시사점**
- 같은 질문을 같은 설정으로 보내도 답이 **매번 같다는 보장이 없습니다.** 위의 "서로 다른 답 ○개 / 5"가 그 결과입니다. 글자 하나만 달라도 다른 답으로 세므로, 답을 눈으로 비교해 **어디가 달라졌는지**도 봅니다.
- 출력 토큰 수도 매번 달라질 수 있으니, 길이 · 비용도 한 번 잰 값으로 단정하지 않습니다.
- 출력 토큰에는 보이지 않는 생각(2절)도 들어 있습니다. 그래서 답 글 길이가 비슷해도 출력 토큰은 크게 다를 수 있습니다.
- 그래서 **한 번 돌려 본 결과로 "이 프롬프트는 된다/안 된다"를 판단하면 안 됩니다** — 4세션(평가)의 출발점입니다.

### temperature — 흔들림을 조절하는 손잡이였던 것
`temperature`는 다음 토큰을 얼마나 고르게 뽑을지 정하는 값입니다. **0 = 가장 확률 높은 토큰 위주(덜 흔들림)**, **1 = 더 고르게(더 다양함)**. 위에서 본 흔들림을 줄이려고 예전에는 이 값을 0으로 낮췄습니다.
- **SDK 1.x**: `messages.create()`에서 `temperature` 인자가 빠졌습니다. (`llm.call`은 `extra_body`로 우회해 보냅니다.)
- **API**: 이 과정에서 쓰는 모델(Haiku 5.5 · Sonnet 5.5)은 **1이 아닌 값을 거부합니다** — 400 오류. 1은 받지만 기본값과 같아서 아무것도 바뀌지 않습니다. (이전 세대 Haiku 4.5는 0~1 사이 값을 받았습니다.)

**▶ 이 셀에서 할 것** — 기본 모델(`llm.MODEL`)과 고급 모델(`llm.MODEL_ADVANCED`)에 같은 슬로건 질문을 temperature **0**과 **1**로 각각 보내, 어느 쪽이 받아들여지고 어느 쪽이 거부되는지 봅니다.

In [ ]:
Q = "신제품 무선 청소기 슬로건 하나만 지어줘. 슬로건만 답해."

for model in [llm.MODEL, llm.MODEL_ADVANCED]:
    print("모델:", model)
    for t in [0.0, 1.0]:
        try:
            r = llm.call(Q, model=model, temperature=t, max_tokens=500)
            print(f"   temperature={t}: 받아들였습니다 → {r.text.strip()[:40]}")
        except Exception as e:
            print(f"   temperature={t}: 거부 — {type(e).__name__}: {str(e)[:120]}")
    print()

**✔ 이 셀의 시사점**
- 두 모델 모두 **0은 거부**(400 오류)하고 **1만 받습니다.** 1은 기본값이라, 흔들림을 줄이는 데는 아무 도움이 안 됩니다.
- 즉 지금 모델에서는 흔들림을 손잡이로 줄일 수 없습니다. 그래서 **여러 번 돌려 측정하는 습관(4세션)**이 필요합니다.

> 대신 생긴 조절 손잡이는 `effort`("low"~"max")처럼 *얼마나 생각할지*입니다. 흔들림을 줄이는 손잡이는 아닙니다. 지원 여부는 모델마다 다르므로 강의 당일 문서로 확인합니다.

## 6. 토큰과 비용
비용 = 입력 토큰 × 입력 단가 + 출력 토큰 × 출력 단가. **단가는 강의 당일 요금표로 `common/llm.py`의 `PRICES`를 채웁니다.**
(채우지 않은 모델은 "단가 미입력"으로 나옵니다.)

**▶ 이 셀에서 할 것** — 긴 사내 문서(`11_정보보호지침_개정_긴문서.txt`) 한 편을 세 줄로 요약시킵니다. 받은 블록 종류를 보고, 입력 비용과 출력 비용을 **따로** 계산해 어느 쪽이 큰지 비교한 뒤, 1건 비용과 하루 1,000건일 때의 비용을 봅니다.

In [ ]:
doc = Path("data/docs/11_정보보호지침_개정_긴문서.txt").read_text(encoding="utf-8")
r = llm.call(f"다음 문서를 세 줄로 요약해 주세요.\n\n{doc}", max_tokens=2000)
print(r.text.strip(), "\n")
print("받은 블록:", [block.type for block in r.raw.content])

price = llm.PRICES.get(llm.MODEL)              # (입력 단가, 출력 단가) — 100만 토큰당 달러
if price is None:
    print("단가 미입력 — common/llm.py의 PRICES를 채웁니다")
else:
    cost_in = r.input_tokens * price[0] / 1_000_000
    cost_out = r.output_tokens * price[1] / 1_000_000
    print(f"입력 {r.input_tokens:>5} 토큰 × ${price[0]} → {llm.fmt_cost(cost_in)}")
    print(f"출력 {r.output_tokens:>5} 토큰 × ${price[1]} → {llm.fmt_cost(cost_out)}")
    print(f"1건 합계 {llm.fmt_cost(r.cost)} · 하루 1,000건이면 약 ${r.cost * 1000:.2f}")

**✔ 이 셀의 시사점**
- 긴 문서를 넣으면 **입력 토큰이 출력 토큰보다 몇 배 많습니다.** 그런데 출력 단가가 입력의 5배라서, 입력 비용과 출력 비용은 **비슷해지거나 출력 쪽이 더 클 수도** 있습니다. 위 두 줄을 비교해 봅니다.
- 출력 토큰에는 보이지 않는 생각(2절)도 들어갑니다. 받은 블록에 `thinking`이 있었다면, 세 줄 요약치고 출력 토큰이 큰 이유가 그것입니다.
- 1건은 작아 보여도 건수를 곱하면 커집니다. 그래서 비용을 줄이려면 **넣는 양(입력)과 쓰게 하는 양(출력)을 둘 다** 봐야 합니다. 넣는 양은 내일 1세션(전부 넣기 vs 골라 넣기)에서 이어서 봅니다.